# Async Batch Prediction Example

This notebook demonstrates:
1. Using the async client for non-blocking operations
2. Concurrent processing of multiple batches
3. Async polling and result retrieval

## Setup

Import required modules and initialize the async client.

In [ ]:
import os
import asyncio

from spatialise import AsyncSpatialiseSoilPrediction

print("✓ Async modules imported")

## Helper Functions

Define async functions for batch creation and polling.

In [ ]:
async def create_and_poll_batch(client, jobs, metadata=None, poll_interval=5):
    """Create a batch and poll until completion.

    Args:
        client: AsyncSpatialiseSoilPrediction client
        jobs: List of prediction jobs
        metadata: Optional metadata for the batch
        poll_interval: Seconds between status checks

    Returns:
        Final batch status
    """
    # Create batch
    print(f"Creating batch with {len(jobs)} jobs...")
    batch = await client.batch.create(jobs=jobs, metadata=metadata)

    print(f"✓ Batch created: {batch.batch_id}")
    print(f"  Status: {batch.status}")
    print()

    # Poll for completion
    print(f"Polling batch {batch.batch_id}...")
    while True:
        status = await client.batch.retrieve_status(batch.batch_id)

        print(f"  {status.status}: {status.completed_jobs}/{status.total_jobs} completed, {status.failed_jobs} failed")

        if status.status in ["completed", "failed"]:
            return status

        await asyncio.sleep(poll_interval)


async def process_batch(client, batch_name, jobs, metadata):
    """Process a single batch from creation to completion."""
    print(f"\n{'=' * 60}")
    print(f"Processing: {batch_name}")
    print(f"{'=' * 60}")

    try:
        status = await create_and_poll_batch(client, jobs, metadata)

        print(f"\n✓ {batch_name} completed!")
        print(f"  Total jobs: {status.total_jobs}")
        print(f"  Completed: {status.completed_jobs}")
        print(f"  Failed: {status.failed_jobs}")

        # Display results
        for job in status.jobs:
            if job.status == "completed" and job.signed_cog_url:
                print(f"  ✓ Result available: {job.signed_cog_url[:80]}...")
            elif job.status == "failed":
                print(f"  ✗ Job failed: {job.error_message}")

        return status

    except Exception as e:
        print(f"✗ Error processing {batch_name}: {e}")
        raise


print("✓ Async helper functions defined")

## Example 1: Single Batch (Async)

Create and process a single batch using the async client.

In [ ]:
async def single_batch_example():
    """Process a single batch asynchronously."""
    async with AsyncSpatialiseSoilPrediction(
        api_key=os.environ.get("SPATIALISE_API_KEY"),
    ) as client:
        jobs = [
            {
                "year": 2021,
                "polygon": {
                    "type": "Polygon",
                    "coordinates": [
                        [
                            [6.7, 52.8],
                            [6.70074, 52.8],
                            [6.70074, 52.80045],
                            [6.7, 52.80045],
                            [6.7, 52.8],
                        ]
                    ],
                },
            },
            {
                "year": 2022,
                "polygon": {
                    "type": "Polygon",
                    "coordinates": [
                        [
                            [6.7, 52.8],
                            [6.70074, 52.8],
                            [6.70074, 52.80045],
                            [6.7, 52.80045],
                            [6.7, 52.8],
                        ]
                    ],
                },
            },
        ]

        status = await process_batch(
            client,
            "Single Batch Example",
            jobs,
            {"location": "Netherlands", "project": "Async Example"},
        )

        return status


# Run the single batch example
result = await single_batch_example()
print(f"\n✓ Single batch example completed")

## Example 2: Concurrent Multiple Batches

Process multiple batches concurrently for maximum efficiency.

In [ ]:
async def concurrent_batches_example():
    """Process multiple batches concurrently."""
    async with AsyncSpatialiseSoilPrediction(
        api_key=os.environ.get("SPATIALISE_API_KEY"),
    ) as client:
        # Define two different areas of interest
        area_1 = {
            "type": "Polygon",
            "coordinates": [
                [
                    [6.7, 52.8],
                    [6.70074, 52.8],
                    [6.70074, 52.80045],
                    [6.7, 52.80045],
                    [6.7, 52.8],
                ]
            ],
        }

        area_2 = {
            "type": "Polygon",
            "coordinates": [
                [
                    [6.75, 52.85],
                    [6.75074, 52.85],
                    [6.75074, 52.85045],
                    [6.75, 52.85045],
                    [6.75, 52.85],
                ]
            ],
        }

        # Create multiple batches to process concurrently
        batch_1_jobs = [
            {"year": 2021, "polygon": area_1},
            {"year": 2022, "polygon": area_1},
        ]

        batch_2_jobs = [
            {"year": 2021, "polygon": area_2},
            {"year": 2022, "polygon": area_2},
        ]

        # Process both batches concurrently using asyncio.gather
        results = await asyncio.gather(
            process_batch(
                client,
                "Batch 1 - Area North",
                batch_1_jobs,
                {"location": "Netherlands North", "project": "Async Example"},
            ),
            process_batch(
                client,
                "Batch 2 - Area South",
                batch_2_jobs,
                {"location": "Netherlands South", "project": "Async Example"},
            ),
            return_exceptions=True,  # Continue even if one batch fails
        )

        return results


# Run the concurrent batches example
results = await concurrent_batches_example()

## Summary

Display a summary of all processed batches.

In [ ]:
print(f"\n{'=' * 60}")
print("Summary:")
print(f"{'=' * 60}")

for i, result in enumerate(results, 1):
    if isinstance(result, Exception):
        print(f"  Batch {i}: ✗ Failed - {result}")
    else:
        print(f"  Batch {i}: ✓ Completed ({result.completed_jobs}/{result.total_jobs} jobs)")
        print(f"    Batch ID: {result.batch_id}")
        print(f"    Created: {result.created_at}")
        print(f"    Updated: {result.updated_at}")

## Advanced: Using aiohttp Backend

For improved concurrency performance, you can use aiohttp instead of httpx.

In [ ]:
# First install aiohttp support:
# !pip install --pre 'spatialise[aiohttp]'

from spatialise import DefaultAioHttpClient


async def aiohttp_example():
    """Use aiohttp backend for improved concurrency."""
    async with AsyncSpatialiseSoilPrediction(
        api_key=os.environ.get("SPATIALISE_API_KEY"),
        http_client=DefaultAioHttpClient(),  # Use aiohttp instead of httpx
    ) as client:
        batch = await client.batch.create(
            jobs=[
                {
                    "year": 2021,
                    "polygon": {
                        "type": "Polygon",
                        "coordinates": [
                            [
                                [6.7, 52.8],
                                [6.70074, 52.8],
                                [6.70074, 52.80045],
                                [6.7, 52.80045],
                                [6.7, 52.8],
                            ]
                        ],
                    },
                }
            ],
        )

        print(f"✓ Batch created with aiohttp backend: {batch.batch_id}")
        return batch


# Uncomment to run:
# batch = await aiohttp_example()
print("\n💡 Tip: Install 'spatialise[aiohttp]' for improved async performance")